In [36]:
import joblib

# Load variables back into memory
data_dump = joblib.load('processed_data.pkl')
X_train_scaled = data_dump['X_train_scaled']
X_test_scaled = data_dump['X_test_scaled']
y_clf_train = data_dump['y_clf_train']
y_clf_test = data_dump['y_clf_test']

In [37]:
import pandas as pd 
import numpy as np 
import os 

#1. Load the cleaned.csv file
df = pd.read_csv(r"C:\Users\181ci\Masai AI_ML\Capestone Project\cleaned_data.csv")
print("Loaded sucessfully")

Loaded sucessfully


In [38]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load the exact same dataset
df = pd.read_csv(r"C:\Users\181ci\Masai AI_ML\Capestone Project\cleaned_data.csv")

# 2. Define the binarized classification target (high_profit)
y_reg = df['profit']
median_profit = y_reg.median()
y_clf = (y_reg > median_profit).astype(int)

# 3. Define features matching your setup
X = df[['quantity', 'unit_price', 'discount_pct']]

# 4. Leak-free train-test split (using the exact same random_state)
X_train, X_test, y_clf_train, y_clf_test = train_test_split(
    X, y_clf, test_size=0.2, random_state=42
)

# 5. Apply Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Decision Tree baseline

In [39]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Instantiate the model with no hyperparameter constraints
dt_model = DecisionTreeClassifier(max_depth=None, random_state=42)

# 2. Train the model
dt_model.fit(X_train_scaled, y_clf_train)

# 3. Predict classifications
y_pred_train = dt_model.predict(X_train_scaled)
y_pred_test = dt_model.predict(X_test_scaled)

# 4. Calculate accuracies
train_accuracy = accuracy_score(y_clf_train, y_pred_train)
test_accuracy = accuracy_score(y_clf_test, y_pred_test)

print(f"--- Decision Tree Results ---")
print(f"Training Accuracy: {train_accuracy * 100:.2f}%")
print(f"Test Accuracy:     {test_accuracy * 100:.2f}%")

--- Decision Tree Results ---
Training Accuracy: 100.00%
Test Accuracy:     83.00%


# Controlled Decision Tree

In [40]:
# 1. Instantiate the second model with constraints
dt_model_controlled = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)

# 2. Train the model
dt_model_controlled.fit(X_train_scaled, y_clf_train)

# 3. Predict classifications
y_pred_train_controlled = dt_model_controlled.predict(X_train_scaled)
y_pred_test_controlled = dt_model_controlled.predict(X_test_scaled)

# 4. Calculate accuracies
train_accuracy_controlled = accuracy_score(y_clf_train, y_pred_train_controlled)
test_accuracy_controlled = accuracy_score(y_clf_test, y_pred_test_controlled)

print("--- Controlled Decision Tree Results ---")
print(f"Training Accuracy: {train_accuracy_controlled * 100:.2f}%")
print(f"Test Accuracy:     {test_accuracy_controlled * 100:.2f}%")

--- Controlled Decision Tree Results ---
Training Accuracy: 94.25%
Test Accuracy:     80.00%


In [43]:
print(f"\n--- Comparison: Unconstrained(Decision Tree) vs Controlled Decision Tree ---")
print(f"{'Metric':<30} {'Unconstrained':<18} {'Controlled':<18}")
print("-" * 66)
print(f"{'Training Accuracy':<30} {train_accuracy:<18.4f} {train_accuracy_controlled:<18.4f}")
print(f"{'Test Accuracy':<30} {test_accuracy:<18.4f} {test_accuracy_controlled:<18.4f}")
print(f"{'Train-Test Gap':<30} {train_accuracy - test_accuracy:<18.4f} {train_accuracy_controlled - test_accuracy_controlled:<18.4f}")


--- Comparison: Unconstrained vs Controlled Decision Tree ---
Metric                         Unconstrained      Controlled        
------------------------------------------------------------------
Training Accuracy              1.0000             0.9425            
Test Accuracy                  0.8300             0.8000            
Train-Test Gap                 0.1700             0.1425            


# Gini vs Entropy comparison

In [44]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Train DecisionTreeClassifier with Gini Impurity criterion
dt_gini = DecisionTreeClassifier(criterion='gini', max_depth=5, random_state=42)
dt_gini.fit(X_train_scaled, y_clf_train)

# 2. Train DecisionTreeClassifier with Information Gain (Entropy) criterion
dt_entropy = DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=42)
dt_entropy.fit(X_train_scaled, y_clf_train)

# 3. Predict classifications on the test data
y_pred_gini_test = dt_gini.predict(X_test_scaled)
y_pred_entropy_test = dt_entropy.predict(X_test_scaled)

# 4. Calculate and report test accuracies
test_acc_gini = accuracy_score(y_clf_test, y_pred_gini_test)
test_acc_entropy = accuracy_score(y_clf_test, y_pred_entropy_test)

print("--- Decision Tree Structural Criteria Comparison ---")
print(f"Gini Criterion Test Accuracy   : {test_acc_gini * 100:.2f}%")
print(f"Entropy Criterion Test Accuracy: {test_acc_entropy * 100:.2f}%")

--- Decision Tree Structural Criteria Comparison ---
Gini Criterion Test Accuracy   : 80.00%
Entropy Criterion Test Accuracy: 79.00%


# Random Forest

In [45]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Assuming X_train_scaled, X_test_scaled, y_clf_train, y_clf_test are already defined 
# feature_names = X_train.columns

# 1. Train the Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train_scaled, y_clf_train)

# 2. Predict probabilities and classes
y_pred_train = rf_model.predict(X_train_scaled)
y_pred_test = rf_model.predict(X_test_scaled)

# Predict probabilities for the positive class (assumed to be 1) for ROC-AUC
y_prob_train = rf_model.predict_proba(X_train_scaled)[:, 1]
y_prob_test = rf_model.predict_proba(X_test_scaled)[:, 1]

# 3. Calculate metrics
train_accuracy = accuracy_score(y_clf_train, y_pred_train)
test_accuracy = accuracy_score(y_clf_test, y_pred_test)

# Calculate ROC-AUC (handle binary or multi-class appropriately; assumes binary here)
train_auc = roc_auc_score(y_clf_train, y_prob_train)
test_auc = roc_auc_score(y_clf_test, y_prob_test)

print("--- Random Forest Classifier Results ---")
print(f"Training Accuracy : {train_accuracy * 100:.2f}%")
print(f"Test Accuracy     : {test_accuracy * 100:.2f}%")
print(f"Training ROC-AUC  : {train_auc:.4f}")
print(f"Test ROC-AUC      : {test_auc:.4f}\n")

# 4. Retrieve and display top 5 feature importances
importances = rf_model.feature_importances_

# If feature names aren't explicitly saved, generate generic ones based on column count
if 'feature_names' not in locals():
    feature_names = [f"Feature {i}" for i in range(X_train_scaled.shape[1])]

# Create a sorted DataFrame
feature_imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("--- Top 5 Features by Importance ---")
print(feature_imp_df.head(5).to_string(index=False))

--- Random Forest Classifier Results ---
Training Accuracy : 99.75%
Test Accuracy     : 82.00%
Training ROC-AUC  : 1.0000
Test ROC-AUC      : 0.9346

--- Top 5 Features by Importance ---
  Feature  Importance
Feature 1    0.735308
Feature 0    0.158053
Feature 2    0.106640


# Gradient Boosting

In [46]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# 1. Instantiate the Gradient Boosting Classifier with specified hyperparameters
gb_model = GradientBoostingClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=3, 
    random_state=42
)

# 2. Train the model
gb_model.fit(X_train_scaled, y_clf_train)

# 3. Predict classifications
y_pred_train = gb_model.predict(X_train_scaled)
y_pred_test = gb_model.predict(X_test_scaled)

# 4. Predict probabilities for the positive class (needed for ROC-AUC)
y_prob_train = gb_model.predict_proba(X_train_scaled)[:, 1]
y_prob_test = gb_model.predict_proba(X_test_scaled)[:, 1]

# 5. Calculate accuracies and ROC-AUC scores
train_accuracy = accuracy_score(y_clf_train, y_pred_train)
test_accuracy = accuracy_score(y_clf_test, y_pred_test)

train_auc = roc_auc_score(y_clf_train, y_prob_train)
test_auc = roc_auc_score(y_clf_test, y_prob_test)

print(f"--- Gradient Boosting Results ---")
print(f"Training Accuracy: {train_accuracy * 100:.2f}%")
print(f"Test Accuracy:     {test_accuracy * 100:.2f}%")
print(f"Training ROC-AUC:  {train_auc:.4f}")
print(f"Test ROC-AUC:      {test_auc:.4f}")

--- Gradient Boosting Results ---
Training Accuracy: 99.50%
Test Accuracy:     80.00%
Training ROC-AUC:  1.0000
Test ROC-AUC:      0.9310


# Feature ablation study

In [49]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# --- Phase 1: Re-evaluate Full Model & Get Importances ---
# Using the identical hyperparameters and random_state from Task 4
rf_full = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_full.fit(X_train_scaled, y_clf_train)

# Calculate test AUC for the full model
y_prob_test_full = rf_full.predict_proba(X_test_scaled)[:, 1]
auc_full_features = roc_auc_score(y_clf_test, y_prob_test_full)

# Get feature importances
importances = rf_full.feature_importances_

# --- Phase 2: Identify and Drop the 5 Lowest-Importance Features ---
# Check if we have enough features to remove 5
total_features = X_train_scaled.shape[1]
features_to_remove = min(5, max(1, total_features - 1))  # Remove at most 5, but leave at least 1 feature

print(f"Total features: {total_features}")
print(f"Features to remove: {features_to_remove}")

# Get indices of features sorted from lowest to highest importance
sorted_indices = np.argsort(importances)
lowest_indices = sorted_indices[:features_to_remove]  # Use dynamic number instead of fixed 5

# Create reduced datasets by dropping these feature columns
X_train_reduced = np.delete(X_train_scaled, lowest_indices, axis=1)
X_test_reduced = np.delete(X_test_scaled, lowest_indices, axis=1)

# Verify we still have features left
print(f"Features remaining after reduction: {X_train_reduced.shape[1]}")

# --- Phase 3: Train and Evaluate the Reduced Model ---
rf_reduced = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_reduced.fit(X_train_reduced, y_clf_train)

# Calculate test AUC for the reduced model
y_prob_test_reduced = rf_reduced.predict_proba(X_test_reduced)[:, 1]
auc_reduced_features = roc_auc_score(y_clf_test, y_prob_test_reduced)

# --- Phase 4: Report Comparative Metrics ---
print("--- Advanced Analysis: Feature Reduction Experiment ---")
print(f"(a) Test-set ROC-AUC (Full Model - All Features): {auc_full_features:.4f}")
print(f"(b) Test-set ROC-AUC (Reduced Model - {features_to_remove} Features Dropped): {auc_reduced_features:.4f}")
print(f"Absolute AUC Difference: {auc_reduced_features - auc_full_features:.4f}")

Total features: 3
Features to remove: 2
Features remaining after reduction: 1
--- Advanced Analysis: Feature Reduction Experiment ---
(a) Test-set ROC-AUC (Full Model - All Features): 0.9346
(b) Test-set ROC-AUC (Reduced Model - 2 Features Dropped): 0.8121
Absolute AUC Difference: -0.1224


# Cross-validated comparison

In [50]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# 1. Define the unified Stratified 5-Fold cross-validation driver
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 2. Instantiate all 4 controlled classifier profiles
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree (Max Depth: 5)": DecisionTreeClassifier(criterion='gini', max_depth=5, random_state=42),
    "Random Forest (100 Trees)": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    "Gradient Boosting Machine": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}

print("--- 5-Fold Stratified Cross-Validation Performance (ROC-AUC) ---")

# 3. Iterate, score, and calculate performance metrics
for name, model in models.items():
    # Evaluate using the full scaled dataset (X_train_scaled combined with X_test_scaled) 
    # or X_train_scaled directly to evaluate pipeline consistency
    auc_scores = cross_val_score(
        model, 
        X_train_scaled, 
        y_clf_train, 
        cv=cv_strategy, 
        scoring='roc_auc',
        n_jobs=-1
    )
    
    mean_auc = np.mean(auc_scores)
    std_auc = np.std(auc_scores)
    
    print(f"{name.ljust(30)} -> Mean AUC: {mean_auc:.4f} | Std Dev: ±{std_auc:.4f}")

--- 5-Fold Stratified Cross-Validation Performance (ROC-AUC) ---
Logistic Regression            -> Mean AUC: 0.9475 | Std Dev: ±0.0102
Decision Tree (Max Depth: 5)   -> Mean AUC: 0.9523 | Std Dev: ±0.0101
Random Forest (100 Trees)      -> Mean AUC: 0.9705 | Std Dev: ±0.0081
Gradient Boosting Machine      -> Mean AUC: 0.9767 | Std Dev: ±0.0097


# Hyperparameter tuning with GridSearchCV

In [51]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# 1. Define the parameter grid (using double underscores for pipeline parameters)
param_grid = {
    'randomforestclassifier__n_estimators': [50, 100, 200],
    'randomforestclassifier__max_depth': [5, 10, None],
    'randomforestclassifier__min_samples_leaf': [1, 5]
}

# 2. Build the automated preprocessing and classification Pipeline
pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

# 3. Configure the Stratified 5-Fold cross-validation driver
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 4. Instantiate the Grid Search object
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1
)

# 5. Execute search on raw, unscaled features (assuming X_train, y_clf_train match your pipeline data)
print("Executing GridSearchCV across parameter space...")
grid_search.fit(X_train, y_clf_train)

# 6. Report the optimal discovery metrics
print("\n--- Hyperparameter Optimization Results ---")
print(f"Best Parameters Found: {grid_search.best_params_}")
print(f"Best Cross-Validated ROC-AUC Score: {grid_search.best_score_:.4f}")

Executing GridSearchCV across parameter space...

--- Hyperparameter Optimization Results ---
Best Parameters Found: {'randomforestclassifier__max_depth': 10, 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 100}
Best Cross-Validated ROC-AUC Score: 0.9695


# Manual learning curve

In [52]:
# 1. Extract the best estimator pipeline from your tuned GridSearchCV object
best_pipeline = grid_search.best_estimator_

# 2. Define progressive evaluation fractions
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
results = []

print("Generating learning curve metrics...")

# 3. Step sequentially through data subsets
for f in fractions:
    # Compute row threshold index
    subset_size = int(f * len(X_train))
    
    # Slice the first 'subset_size' rows from raw unscaled training pairs
    X_train_sub = X_train.iloc[:subset_size] if hasattr(X_train, 'iloc') else X_train[:subset_size]
    y_train_sub = y_clf_train.iloc[:subset_size] if hasattr(y_clf_train, 'iloc') else y_clf_train[:subset_size]
    
    # Fit the full pipeline on the subset (Pipeline handles Imputer -> Scaler -> RF internally)
    best_pipeline.fit(X_train_sub, y_train_sub)
    
    # Predict probabilities on the current training subset
    y_prob_train = best_pipeline.predict_proba(X_train_sub)[:, 1]
    
    # Predict probabilities on the fixed full test set (unscaled raw X_test since pipeline scales)
    y_prob_test = best_pipeline.predict_proba(X_test)[:, 1]
    
    # Calculate ROC-AUC metrics
    train_auc = roc_auc_score(y_train_sub, y_prob_train)
    test_auc = roc_auc_score(y_clf_test, y_prob_test)
    
    # Append values
    results.append({
        "Training Fraction": f"{int(f*100)}%",
        "Training AUC": f"{train_auc:.4f}",
        "Test AUC": f"{test_auc:.4f}"
    })

# 4. Display results as a clean formatted table DataFrame
learning_curve_df = pd.DataFrame(results)
print("\n--- Model Learning Curve Analysis ---")
print(learning_curve_df.to_string(index=False))

Generating learning curve metrics...

--- Model Learning Curve Analysis ---
Training Fraction Training AUC Test AUC
              20%       1.0000   0.9380
              40%       1.0000   0.9213
              60%       1.0000   0.9177
              80%       1.0000   0.9233
             100%       1.0000   0.9346


# Serialize the best model

In [53]:
import joblib

# 1. Extract the best estimator pipeline from your tuned GridSearchCV object
best_pipeline = grid_search.best_estimator_

# 2. Save the fully trained pipeline to disk
joblib.dump(best_pipeline, 'best_model.pkl')
print("Optimal pipeline successfully serialized and saved to 'best_model.pkl'")

Optimal pipeline successfully serialized and saved to 'best_model.pkl'


In [54]:
import joblib
import pandas as pd
import numpy as np

# 1. Reload the optimized pipeline from disk
loaded_pipeline = joblib.load('best_model.pkl')

# 2. Re-create the column structure expected by your pipeline
# (Replace with your actual X_train column names if they differ)
feature_cols = X_train.columns if hasattr(X_train, 'columns') else [f"feature_{i}" for i in range(X_train.shape[1])]

# 3. Create 2 hand-crafted test rows representing hypothetical customer profiles
# (Using np.nan to verify that the internal SimpleImputer fires correctly)
mock_data = [
    [1, 2.5, 45.0, 0.15, 3000.0, np.nan, 2, 1],  # Row 1 with a missing feature value
    [0, 1.2, 12.0, 0.00, 450.0, 150.0, 1, 0]    # Row 2 with complete features
]

# Trim or adjust mock data length to perfectly match feature count
mock_data = [row[:len(feature_cols)] for row in mock_data]

X_mock = pd.DataFrame(mock_data, columns=feature_cols)

# 4. Generate classifications using the reloaded model pipeline
mock_predictions = loaded_pipeline.predict(X_mock)
mock_probabilities = loaded_pipeline.predict_proba(X_mock)[:, 1]

print("--- Pipeline Inference Verification ---")
print(f"Predicted Classes     : {mock_predictions}")
print(f"Positive Probabilities: {mock_probabilities}")

--- Pipeline Inference Verification ---
Predicted Classes     : [0 0]
Positive Probabilities: [0.01 0.01]


In [55]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# 1. Define models using identical hyperparameters from Parts 2 & 3
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Controlled Decision Tree (depth=5)": DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42),
    "Random Forest (Tuned Pipeline)": grid_search.best_estimator_ if 'grid_search' in locals() else 
                                     RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    "Gradient Boosting Machine": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}

summary_data = []

print("Extracting model final metrics...")

for name, model in models.items():
    # A. Calculate 5-Fold Cross-Validation Metrics
    # If using the pipeline, pass raw X_train; if passing a raw estimator, use X_train_scaled
    X_cv = X_train if "Pipeline" in str(type(model)) else X_train_scaled
    
    cv_scores = cross_val_score(model, X_cv, y_clf_train, cv=cv_strategy, scoring='roc_auc', n_jobs=-1)
    mean_cv_auc = np.mean(cv_scores)
    std_cv_auc = np.std(cv_scores)
    
    # B. Calculate Holdout Test-Set Metrics
    # Re-fit on full training subset to find test performance
    model.fit(X_cv, y_clf_train)
    X_t = X_test if "Pipeline" in str(type(model)) else X_test_scaled
    test_probs = model.predict_proba(X_t)[:, 1]
    test_auc = roc_auc_score(y_clf_test, test_probs)
    
    summary_data.append({
        "Model Name": name,
        "5-Fold CV Mean AUC": f"{mean_cv_auc:.4f}",
        "5-Fold CV Std AUC": f"±{std_cv_auc:.4f}",
        "Test-Set AUC": f"{test_auc:.4f}"
    })

# 2. Display results table
summary_df = pd.DataFrame(summary_data)
print("\n--- Summary Performance Matrix ---")
print(summary_df.to_string(index=False))

Extracting model final metrics...

--- Summary Performance Matrix ---
                        Model Name 5-Fold CV Mean AUC 5-Fold CV Std AUC Test-Set AUC
               Logistic Regression             0.9475           ±0.0102       0.9141
Controlled Decision Tree (depth=5)             0.9582           ±0.0089       0.9173
    Random Forest (Tuned Pipeline)             0.9695           ±0.0089       0.9346
         Gradient Boosting Machine             0.9767           ±0.0097       0.9310
